In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/StudentsPerformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
messy_df = df.copy()

# Add missing values
messy_df.loc[0, "math score"] = np.nan
messy_df.loc[1, "reading score"] = np.nan
messy_df.loc[2, "writing score"] = np.nan
messy_df.loc[3, "test preparation course"] = np.nan

# Add duplicate rows
messy_df = pd.concat([messy_df, messy_df.iloc[[0, 1, 2]]], ignore_index=True)

# Add outliers
messy_df.loc[4, "math score"] = 150
messy_df.loc[5, "reading score"] = -10
messy_df.loc[6, "writing score"] = 140

messy_df.to_csv("../data/StudentsPerformance_messy.csv", index=False)

messy_df.head(10)

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,NaN,72.0,74.0
1,female,group C,some college,standard,completed,69.0,NaN,88.0
2,female,group B,master's degree,standard,none,90.0,95.0,NaN
3,male,group A,associate's degree,free/reduced,NaN,47.0,57.0,44.0
4,male,group C,some college,standard,none,150.0,78.0,75.0
5,female,group B,associate's degree,standard,none,71.0,-10.0,78.0
6,female,group B,some college,standard,completed,88.0,95.0,140.0
7,male,group B,some college,free/reduced,none,40.0,43.0,39.0
8,male,group D,high school,free/reduced,completed,64.0,64.0,67.0
9,female,group B,high school,free/reduced,none,38.0,60.0,50.0


In [4]:
messy_df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        1
math score                     2
reading score                  2
writing score                  2
dtype: int64

In [5]:
messy_df.duplicated().sum()

np.int64(3)

In [6]:
numeric_cols = ["math score", "reading score", "writing score"]

outlier_check = messy_df[
    (messy_df[numeric_cols] < 0).any(axis=1) |
    (messy_df[numeric_cols] > 100).any(axis=1)
]

outlier_check

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
4,male,group C,some college,standard,none,150.0,78.0,75.0
5,female,group B,associate's degree,standard,none,71.0,-10.0,78.0
6,female,group B,some college,standard,completed,88.0,95.0,140.0


In [7]:
before_cleaning = {
    "Rows": messy_df.shape[0],
    "Columns": messy_df.shape[1],
    "Missing Values": messy_df.isnull().sum().sum(),
    "Duplicate Rows": messy_df.duplicated().sum(),
    "Outlier Rows": len(outlier_check)
}

before_cleaning

{'Rows': 1003,
 'Columns': 8,
 'Missing Values': np.int64(7),
 'Duplicate Rows': np.int64(3),
 'Outlier Rows': 3}

In [8]:
cleaned_df = messy_df.copy()

# Fill missing numerical values with median
for col in numeric_cols:
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

# Fill missing categorical values with mode
categorical_cols = cleaned_df.select_dtypes(include="object").columns

for col in categorical_cols:
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].mode()[0])

cleaned_df.isnull().sum()

C:\Users\HP\AppData\Local\Temp\ipykernel_2036\4019012638.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = cleaned_df.select_dtypes(include="object").columns


gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [9]:
cleaned_df = cleaned_df.drop_duplicates()

cleaned_df.duplicated().sum()

np.int64(0)

In [10]:
for col in numeric_cols:
    cleaned_df[col] = cleaned_df[col].clip(lower=0, upper=100)

cleaned_df[
    (cleaned_df[numeric_cols] < 0).any(axis=1) |
    (cleaned_df[numeric_cols] > 100).any(axis=1)
]

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score


In [11]:
cleaned_df.to_csv("../data/StudentsPerformance_cleaned.csv", index=False)

cleaned_df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,66.0,72.0,74.0
1,female,group C,some college,standard,completed,69.0,70.0,88.0
2,female,group B,master's degree,standard,none,90.0,95.0,69.0
3,male,group A,associate's degree,free/reduced,none,47.0,57.0,44.0
4,male,group C,some college,standard,none,100.0,78.0,75.0


In [12]:
after_cleaning = {
    "Rows": cleaned_df.shape[0],
    "Columns": cleaned_df.shape[1],
    "Missing Values": cleaned_df.isnull().sum().sum(),
    "Duplicate Rows": cleaned_df.duplicated().sum(),
    "Outlier Rows": len(
        cleaned_df[
            (cleaned_df[numeric_cols] < 0).any(axis=1) |
            (cleaned_df[numeric_cols] > 100).any(axis=1)
        ]
    )
}

comparison = pd.DataFrame({
    "Before Cleaning": before_cleaning,
    "After Cleaning": after_cleaning
})

comparison

,Before Cleaning,After Cleaning
Rows,1003,1000
Columns,8,8
Missing Values,7,0
Duplicate Rows,3,0
Outlier Rows,3,0


## Data Cleaning Summary

The messy dataset contained missing values, duplicate rows, and outliers. Missing numerical values were handled using the median, while missing categorical values were handled using the mode. Duplicate rows were removed. Outlier score values below 0 and above 100 were clipped to the valid score range of 0 to 100. Finally, the cleaned dataset was saved as `StudentsPerformance_cleaned.csv`.